# 05.1 — RAG Combined (Query Rewriting + Context Reranking) dengan OpenAI

Notebook ini mengevaluasi kombinasi dua teknik mitigasi halusinasi dengan model OpenAI.

**Pipeline:**
```
original query
  -> QR (rewrite via OpenAI)
  -> BM25 retrieve top-20 (pakai rewritten query)
  -> CrossEncoder rerank top-5 (pakai rewritten query)
  -> OpenAI generate (pakai ORIGINAL question)
```

**Model:** `gpt-4.1-mini` via OpenAI API
**Evaluator:** Custom zero-NaN 4 metrik (faithfulness, context_recall, answer_relevancy, context_precision)

In [1]:
# Install dependencies (jalankan sekali saja)
# !pip install openai rank-bm25 sentence-transformers datasets

In [2]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from openai import OpenAI
from rank_bm25 import BM25Okapi
from datasets import load_dataset

try:
    from sentence_transformers import CrossEncoder
    print('sentence-transformers tersedia.')
except ImportError:
    print('sentence-transformers belum terinstall!')
    print('Jalankan: pip install sentence-transformers')
    raise

warnings.filterwarnings('ignore')
print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers tersedia.
Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5


In [ ]:
# ============================================================
# PATH SETUP — auto-resolve PROJECT_ROOT
# ============================================================
from pathlib import Path

_HERE = Path('.').resolve()
PROJECT_ROOT = next((p for p in [_HERE] + list(_HERE.parents) if p.name == 'Code TA'), _HERE.parent.parent.parent)
NOTEBOOKS_V2 = PROJECT_ROOT / 'notebooks'
INDEXES_DIR  = NOTEBOOKS_V2 / 'indexes'
RESULTS_DIR  = PROJECT_ROOT / 'results' / '10_bm25'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BM25_INDEX_PATH = INDEXES_DIR / 'pubmedqa_bm25.pkl'
NOTEBOOK_DIR    = INDEXES_DIR  # kompat lama: BM25_INDEX_PATH dan CHROMA_DB_PATH

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'INDEXES_DIR   : {INDEXES_DIR}')
print(f'RESULTS_DIR   : {RESULTS_DIR}')
print(f'BM25 index    : {BM25_INDEX_PATH.name}')


In [3]:
# ============================================================
# KONFIGURASI
# ============================================================
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_KEY_HERE')

LLM_MODEL      = 'gpt-4.1-mini'  # Model OpenAI
RERANKER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

TOP_K_CANDIDATES = 20
TOP_K_RETRIEVAL  = 5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

# NOTEBOOK_DIR    = Path('.')
# BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
# RESULTS_DIR     = Path('../results')
# RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME        = 'combined_openai'
PHASE1_PATH        = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_CUSTOM_PATH = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'
FINAL_CSV_PATH     = RESULTS_DIR / f'{CONFIG_NAME}_results.csv'

print('Konfigurasi:')
print(f'  LLM          : {LLM_MODEL} (via OpenAI API)')
print(f'  Reranker     : {RERANKER_MODEL}')
print(f'  Retriever    : BM25 top-{TOP_K_CANDIDATES} -> CrossEncoder -> top-{TOP_K_RETRIEVAL}')
print(f'  Query Rewrite: ENABLED (pakai {LLM_MODEL})')
print(f'  Sampel       : {MAX_SAMPLES}')
print(f'  Config       : {CONFIG_NAME}')
print(f'  Output P1    : {PHASE1_PATH}')
print(f'  Output P2    : {PHASE2_CUSTOM_PATH}')
print()
if 'YOUR_API_KEY' in OPENAI_API_KEY:
    print('  OPENAI_API_KEY belum diisi! Set env var atau isi di cell ini.')
else:
    print(f'  OPENAI_API_KEY: {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}')

Konfigurasi:
  LLM          : gpt-4.1-mini (via OpenAI API)
  Reranker     : cross-encoder/ms-marco-MiniLM-L-6-v2
  Retriever    : BM25 top-20 -> CrossEncoder -> top-5
  Query Rewrite: ENABLED (pakai gpt-4.1-mini)
  Sampel       : 500
  Config       : combined_openai
  Output P1    : ..\results\combined_openai_phase1_answers.json
  Output P2    : ..\results\combined_openai_phase2_custom.json

  OPENAI_API_KEY: sk-proj-...REDACTED


In [4]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str


@dataclass
class RetrievalResult:
    document       : Document
    score          : float           # BM25 score
    reranker_score : float = 0.0     # CrossEncoder score (diisi setelah rerank)


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


In [5]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    """Muat BM25 index dari file jika ada, atau bangun dari scratch."""
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index baru...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


# Load dataset + BM25 index (shared dengan notebook lain)
_full_data            = load_dataset(DATASET_NAME, DATASET_SUBSET, trust_remote_code=True)['train']
bm25_index, documents = load_or_build_bm25(_full_data.select(range(500)))
pubmedqa_data         = _full_data.select(range(MAX_SAMPLES))
print(f'\nEvaluasi akan menggunakan {len(pubmedqa_data)} sampel pertama.')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen

Evaluasi akan menggunakan 500 sampel pertama.


In [6]:
print(f'Memuat CrossEncoder: {RERANKER_MODEL}...')
print('(Download ~85MB sekali, lalu di-cache)')
t0 = time.time()
cross_encoder = CrossEncoder(RERANKER_MODEL)
print(f'CrossEncoder siap dalam {time.time()-t0:.1f} detik')

# Smoke test
_pairs = [
    ('Does aspirin prevent heart attacks?', 'Aspirin reduces platelet aggregation and is used in cardiovascular prevention.'),
    ('Does aspirin prevent heart attacks?', 'Weather patterns affect agricultural yields in tropical regions.'),
]
_scores = cross_encoder.predict(_pairs)
print(f'\nSmoke test CrossEncoder:')
print(f'  Relevan       : {_scores[0]:.4f}')
print(f'  Tidak relevan : {_scores[1]:.4f}')
assert _scores[0] > _scores[1], 'CrossEncoder gagal membedakan relevan vs tidak!'
print('Reranker berfungsi dengan benar.')

Memuat CrossEncoder: cross-encoder/ms-marco-MiniLM-L-6-v2...
(Download ~85MB sekali, lalu di-cache)


Loading weights: 100%|████████████████████████| 105/105 [00:02<00:00, 51.32it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CrossEncoder siap dalam 13.3 detik

Smoke test CrossEncoder:
  Relevan       : 4.9414
  Tidak relevan : -11.1675
Reranker berfungsi dengan benar.


In [7]:
# ============================================================
# Setup OpenAI Client
# ============================================================
openai_client = OpenAI(api_key=OPENAI_API_KEY)


def openai_generate(prompt: str, max_tokens: int = 300, temperature: float = TEMPERATURE) -> str:
    """Wrapper OpenAI API dengan retry otomatis."""
    for attempt in range(5):
        try:
            response = openai_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
                seed=SEED,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = (attempt + 1) * 10
                print(f'  [Rate limit] Tunggu {wait}s... (attempt {attempt+1}/5)')
                time.sleep(wait)
            elif '500' in err or '502' in err or '503' in err:
                wait = (attempt + 1) * 5
                print(f'  [Server error] Tunggu {wait}s...')
                time.sleep(wait)
            else:
                print(f'  [OpenAI Error] {type(e).__name__}: {err[:100]}')
                raise
    raise RuntimeError('OpenAI API gagal setelah 5 percobaan.')


# Smoke test
print('Testing OpenAI API...')
_test = openai_generate('Reply with exactly: OK', max_tokens=5)
print(f'Response: {_test!r}')
print('OpenAI client siap!')

Testing OpenAI API...
Response: 'OK'
OpenAI client siap!


In [8]:
QUERY_REWRITE_PROMPT = (
    'You are a query rewriting assistant for a biomedical question-answering system.\n'
    'Rewrite the following medical question to improve retrieval from a PubMed research database.\n\n'
    'Rules:\n'
    '1. Be more specific and add relevant medical/scientific terminology.\n'
    '2. Expand abbreviations (e.g. "MI" -> "myocardial infarction").\n'
    '3. Preserve the original yes/no/maybe answerable intent.\n'
    '4. Output ONLY the rewritten question, no explanations.\n\n'
    'Original question: {query}\n\n'
    'Rewritten question:'
)


def rewrite_query(query: str) -> str:
    """Reformulasi query pakai OpenAI. Fallback ke query asli kalau gagal."""
    prompt = QUERY_REWRITE_PROMPT.format(query=query)
    try:
        rewritten = openai_generate(prompt, max_tokens=150, temperature=0.3)
        rewritten = rewritten.strip().replace('\n', ' ')
        return rewritten if len(rewritten) >= 10 else query
    except Exception as e:
        print(f'  [QR Error] {e} -- pakai query asli')
        return query


# Test
print('Contoh Query Rewriting:')
print('=' * 70)
for q in ['Does aspirin reduce the risk of MI?',
          'Can exercise prevent T2DM?']:
    rw = rewrite_query(q)
    print(f'\nAsli    : {q}')
    print(f'Rewrite : {rw}')

Contoh Query Rewriting:

Asli    : Does aspirin reduce the risk of MI?
Rewrite : Does aspirin reduce the risk of myocardial infarction in patients at risk for cardiovascular disease?

Asli    : Can exercise prevent T2DM?
Rewrite : Can regular physical exercise prevent the onset of type 2 diabetes mellitus?


In [9]:
def retrieve_with_qr_cr(
    query: str,
    k_candidates: int = TOP_K_CANDIDATES,
    k_final: int = TOP_K_RETRIEVAL
) -> Tuple[List[RetrievalResult], str]:
    """
    Pipeline gabungan: QR + BM25 + CR.

    1. QR: rewrite via OpenAI
    2. BM25: top-k_candidates pakai rewritten
    3. CrossEncoder: rerank pakai rewritten, top-k_final
    """
    rewritten = rewrite_query(query)

    tokens     = tokenize_bm25(rewritten)
    scores     = bm25_index.get_scores(tokens)
    top_cands  = np.argsort(scores)[::-1][:k_candidates]
    candidates = [
        RetrievalResult(document=documents[i], score=float(scores[i]))
        for i in top_cands
    ]

    pairs           = [(rewritten, r.document.text) for r in candidates]
    reranker_scores = cross_encoder.predict(pairs)
    for r, rs in zip(candidates, reranker_scores):
        r.reranker_score = float(rs)

    reranked = sorted(candidates, key=lambda r: r.reranker_score, reverse=True)
    return reranked[:k_final], rewritten


# Test
test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r, test_rw = retrieve_with_qr_cr(test_q)
print(f'Query asli : {test_q}')
print(f'Rewrite    : {test_rw}')
print(f'\nTop-{TOP_K_RETRIEVAL} dokumen (QR -> BM25 top-{TOP_K_CANDIDATES} -> CR):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] BM25={r.score:.2f} | Reranker={r.reranker_score:.4f} | {r.document.section_label} | {r.document.text[:70]}...')

Query asli : Does aspirin reduce the risk of myocardial infarction?
Rewrite    : Does aspirin administration reduce the risk of acute myocardial infarction in patients at risk for cardiovascular disease?

Top-5 dokumen (QR -> BM25 top-20 -> CR):
  [1] BM25=27.68 | Reranker=-0.7403 | OBJECTIVE | Myocardial damage that is associated with percutaneous coronary interv...
  [2] BM25=32.44 | Reranker=-1.7563 | BACKGROUND | It has recently been shown that non-high density lipoprotein cholester...
  [3] BM25=26.16 | Reranker=-1.9364 | BACKGROUND | Several studies have shown associations between hyperglycemia and risk...
  [4] BM25=31.66 | Reranker=-1.9540 | METHODS | By use of the Cooperative Cardiovascular Project database (a retrospec...
  [5] BM25=28.33 | Reranker=-2.2302 | BACKGROUND | The role of early revascularization among patients with acute myocardi...


In [10]:
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)


def generate_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    """Generate jawaban via OpenAI. PENTING: pakai ORIGINAL query, bukan rewritten."""
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    return openai_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300,
        temperature=TEMPERATURE
    )


# Test
test_ans = generate_answer(test_q, test_r)
print('Output generation:')
print('-' * 60)
print(test_ans)
print('-' * 60)

Output generation:
------------------------------------------------------------
The provided abstracts do not mention aspirin or its effects on the risk of myocardial infarction. They focus on topics such as myocardial damage related to PCI, lipid predictors of cardiovascular risk, hyperglycemia and cardiovascular disease, and outcomes in myocardial infarction complicated by cardiogenic shock, but none address aspirin use or its impact on myocardial infarction risk.

no
------------------------------------------------------------


In [11]:
def extract_label(answer: str) -> str:
    """Ekstrak prediksi yes/no/maybe dari teks jawaban."""
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


cases = [
    ('Strong evidence.\nyes', 'yes'),
    ('No effect found.\nno',  'no'),
    ('Mixed results.\nmaybe', 'maybe'),
    ('Verdict: yes.',          'yes'),
    ('Totally unclear.',       'maybe'),
]
all_ok = all(extract_label(txt) == exp for txt, exp in cases)
print(f'Unit test extract_label: {"PASS" if all_ok else "FAIL"}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label: PASS
Label dari test answer: 'no'


In [12]:
def _split_sentences(text: str) -> List[str]:
    """Pecah teks menjadi kalimat. Filter kalimat terlalu pendek (<15 char)."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt: str) -> bool:
    """Tanya LLM ya/tidak via OpenAI. Return True=yes, False=no. Fallback False jika gagal."""
    try:
        resp = openai_generate(prompt, max_tokens=10, temperature=0.0)
        return 'yes' in resp.lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement directly supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    supported = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return supported / len(sentences)


def compute_context_recall(reference: str, contexts: List[str]) -> float:
    sentences = _split_sentences(reference)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    covered = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s)))
    return covered / len(sentences)


def compute_answer_relevancy(question: str, answer: str) -> float:
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement relevant to answering the question above? '
        'Answer with only "yes" or "no".'
    )
    relevant = sum(1 for s in sentences if _llm_yes_no(prompt_tmpl.format(question=question, sent=s)))
    return relevant / len(sentences)


def compute_context_precision(question: str, contexts: List[str], reference: str) -> float:
    if not contexts:
        return 0.0
    prompt_tmpl = (
        'Question: {question}\n\n'
        'Ground truth answer: {reference}\n\n'
        'Retrieved context: {ctx}\n\n'
        'Does this context contain information useful for correctly answering '
        'the question based on the ground truth? Answer with only "yes" or "no".'
    )
    relevance = []
    for ctx in contexts:
        is_rel = _llm_yes_no(prompt_tmpl.format(
            question=question, reference=reference[:300], ctx=ctx[:400]
        ))
        relevance.append(1 if is_rel else 0)
    total_relevant = sum(relevance)
    if total_relevant == 0:
        return 0.0
    precision_sum = 0.0
    relevant_count = 0
    for k, rel in enumerate(relevance):
        if rel:
            relevant_count += 1
            precision_sum += relevant_count / (k + 1)
    return precision_sum / total_relevant


def evaluate_custom(question: str, answer: str,
                    contexts: List[str], reference: str) -> Dict:
    return {
        'faithfulness'      : compute_faithfulness(answer, contexts),
        'context_recall'    : compute_context_recall(reference, contexts),
        'answer_relevancy'  : compute_answer_relevancy(question, answer),
        'context_precision' : compute_context_precision(question, contexts, reference),
    }


# Smoke test
_ctx = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_ans = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_ref = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_q   = 'Does aspirin prevent heart attacks?'
_r   = evaluate_custom(_q, _ans, _ctx, _ref)
print('Smoke test evaluate_custom (4 metrik):')
print(f'  faithfulness      = {_r["faithfulness"]:.3f}')
print(f'  context_recall    = {_r["context_recall"]:.3f}')
print(f'  answer_relevancy  = {_r["answer_relevancy"]:.3f}')
print(f'  context_precision = {_r["context_precision"]:.3f}')
print('Zero-NaN evaluator siap (4 metrik).')

Smoke test evaluate_custom (4 metrik):
  faithfulness      = 1.000
  context_recall    = 1.000
  answer_relevancy  = 1.000
  context_precision = 1.000
Zero-NaN evaluator siap (4 metrik).


## Demo — 5 Sampel Pertama

In [ ]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama ({LLM_MODEL}, QR+CR)')
print('=' * 65)

for i in range(DEMO_SIZE):
    s         = pubmedqa_data[i]
    q         = s['question']
    gt        = s['final_decision']

    retrieved, rewritten = retrieve_with_qr_cr(q)
    answer               = generate_answer(q, retrieved)
    predicted            = extract_label(answer)
    correct              = predicted == gt

    demo_results.append({
        'idx': i, 'question': q, 'rewritten_query': rewritten,
        'ground_truth': gt, 'predicted_label': predicted,
        'is_correct': correct, 'answer': answer,
    })

    verdict = 'BENAR' if correct else 'SALAH'
    print(f'\n[{i+1}/{DEMO_SIZE}] {q[:75]}...')
    print(f'  Rewritten: {rewritten[:80]}...')
    print(f'  GT={gt} | Pred={predicted} | {verdict}')
    print(f'  Jawaban  : {answer[:120]}...')

n_ok = sum(r['is_correct'] for r in demo_results)
print(f'\n{"="*65}')
print(f'Demo Accuracy: {n_ok}/{DEMO_SIZE} = {n_ok/DEMO_SIZE:.1%}')

## Phase 1 — Generate Jawaban (500 Sampel)

Estimasi waktu GPT-4.1-mini: ~10-20 menit untuk 500 sampel (QR tambah overhead).
Resume otomatis jika interrupted.

In [13]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Memulai Fase 1: {MAX_SAMPLES} sampel (QR + BM25 top-{TOP_K_CANDIDATES} -> CR -> top-{TOP_K_RETRIEVAL}).')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()

    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']

        retrieved, rewritten = retrieve_with_qr_cr(q)
        answer               = generate_answer(q, retrieved)
        predicted            = extract_label(answer)

        phase1_results.append({
            'idx'             : i,
            'pubid'           : str(s['pubid']),
            'question'        : q,
            'rewritten_query' : rewritten,
            'ground_truth'    : gt,
            'predicted_label' : predicted,
            'is_correct'      : predicted == gt,
            'answer'          : answer,
            'contexts'        : [r.document.text for r in retrieved],
            'reference'       : ref,
            'retrieval_scores': [r.score for r in retrieved],
            'reranker_scores' : [r.reranker_score for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({'config': CONFIG_NAME,
                           'llm_model': LLM_MODEL,
                           'timestamp': datetime.now().isoformat(),
                           'max_samples': MAX_SAMPLES, 'completed': i+1,
                           'results': phase1_results}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Akurasi: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')

    print(f'\nFase 1 selesai! Disimpan ke {PHASE1_PATH}')
else:
    print(f'Fase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Memulai Fase 1: 500 sampel (QR + BM25 top-20 -> CR -> top-5).
Memproses 500 sampel tersisa...

  [ 10/500] Akurasi: 60.0% | pred=yes, gt=yes | ETA 36.5 mnt
  [ 20/500] Akurasi: 70.0% | pred=yes, gt=yes | ETA 32.9 mnt
  [ 30/500] Akurasi: 70.0% | pred=yes, gt=yes | ETA 32.7 mnt
  [ 40/500] Akurasi: 62.5% | pred=yes, gt=no | ETA 32.1 mnt
  [ 50/500] Akurasi: 64.0% | pred=no, gt=no | ETA 31.4 mnt
  [ 60/500] Akurasi: 61.7% | pred=yes, gt=yes | ETA 30.2 mnt
  [ 70/500] Akurasi: 62.9% | pred=yes, gt=yes | ETA 29.9 mnt
  [ 80/500] Akurasi: 61.3% | pred=yes, gt=yes | ETA 29.1 mnt
  [ 90/500] Akurasi: 61.1% | pred=no, gt=maybe | ETA 28.5 mnt
  [100/500] Akurasi: 62.0% | pred=yes, gt=yes | ETA 27.5 mnt
  [110/500] Akurasi: 62.7% | pred=yes, gt=yes | ETA 26.7 mnt
  [120/500] Akurasi: 61.7% | pred=yes, gt=yes | ETA 25.9 mnt
  [130/500] Akurasi: 60.0% | pred=yes, gt=maybe | ETA 25.4 mnt
  [140/500] Akurasi: 59.3% | pred=yes, gt=yes | ETA 24.7 mnt
  [150/500] Akurasi: 58.7% | pred=yes, gt=no | ETA 

## Analisis Phase 1

In [14]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']

n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS PHASE 1 — {n} sampel ({CONFIG_NAME})')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')

print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>10}')
print(f'  {"-"*40}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>6} ({g/n:.0%})    | {p:>6} ({p/n:.0%})')

print('\nConfusion Matrix (baris=GT, kolom=Pred):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

# Rata-rata skor BM25 dan Reranker
avg_bm25     = np.mean([np.mean(r['retrieval_scores']) for r in results_p1])
avg_reranker = np.mean([np.mean(r['reranker_scores'])  for r in results_p1])
print(f'\nRata-rata skor BM25     : {avg_bm25:.4f}')
print(f'Rata-rata skor Reranker : {avg_reranker:.4f}')

ANALISIS PHASE 1 — 500 sampel (combined_openai)
Label Accuracy    : 341/500 = 68.2%
Hallucination Rate: 31.8%

  Label    | Ground Truth |   Prediksi
  ----------------------------------------
  yes      |    275 (55%)    |    311 (62%)
  no       |    159 (32%)    |    157 (31%)
  maybe    |     66 (13%)    |     32 (6%)

Confusion Matrix (baris=GT, kolom=Pred):
   GT/Pred     yes      no   maybe
       yes     228      34      13
        no      41     106      12
     maybe      42      17       7

Rata-rata skor BM25     : 27.7300
Rata-rata skor Reranker : -2.3972


## Phase 2 — Custom Evaluator 4 Metrik (500 Sampel)

**Metrik:** Faithfulness, Context Recall, Answer Relevancy, Context Precision.
Estimasi waktu: ~15-30 menit untuk 500 sampel (dengan OpenAI).
Smart resume: deteksi sampel dengan metrik lengkap dan hanya upgrade yang perlu.

In [13]:
MAX_CUSTOM_SAMPLES = 500
REQUIRED_METRICS = ['faithfulness', 'context_recall', 'answer_relevancy', 'context_precision']

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom if all(m in r for m in REQUIRED_METRICS)}
    needs_upgrade = [r for r in p2_custom if not all(m in r for m in REQUIRED_METRICS)]
    print(f'Resume: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai dengan 4 metrik.')
    if needs_upgrade:
        print(f'Perlu upgrade: {len(needs_upgrade)} sampel.')
else:
    p2_custom, done_custom, needs_upgrade = [], set(), []
    print(f'Mulai: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN, 4 metrik).')

# Tahap 1: Upgrade
if needs_upgrade:
    print(f'\nTahap 1: Upgrade {len(needs_upgrade)} sampel...')
    t_up = time.time()
    p1_lookup = {r['idx']: r for r in p1_custom}
    for i, r in enumerate(needs_upgrade):
        src = p1_lookup[r['idx']]
        if 'answer_relevancy' not in r:
            r['answer_relevancy'] = compute_answer_relevancy(src['question'], src['answer'])
        if 'context_precision' not in r:
            r['context_precision'] = compute_context_precision(src['question'], src['contexts'], src['reference'])
        done_custom.add(r['idx'])
        if (i + 1) % 5 == 0 or i == len(needs_upgrade) - 1:
            with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
                json.dump({
                    'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                    'max_samples': MAX_CUSTOM_SAMPLES,
                    'metrics': REQUIRED_METRICS,
                    'evaluator': 'custom_zero_nan_4metrics',
                    'results': p2_custom
                }, f, indent=2, ensure_ascii=False)
            done  = i + 1
            eta   = (time.time()-t_up)/done*(len(needs_upgrade)-done)/60 if done < len(needs_upgrade) else 0
            print(f'  upgrade [{done:3d}/{len(needs_upgrade)}] | ETA {eta:.1f} mnt')

# Tahap 2: Evaluasi sampel baru
remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'\nTahap 2: Evaluasi {len(remaining)} sampel baru...\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({
        'idx': r['idx'], 'ground_truth': r['ground_truth'],
        'predicted_label': r['predicted_label'], 'is_correct': r['is_correct'],
        **scores
    })
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({
                'config': CONFIG_NAME, 'timestamp': datetime.now().isoformat(),
                'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics': REQUIRED_METRICS,
                'evaluator': 'custom_zero_nan_4metrics',
                'results': p2_custom
            }, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f  = sum(x['faithfulness']      for x in p2_custom) / len(p2_custom)
        avg_cr = sum(x['context_recall']    for x in p2_custom) / len(p2_custom)
        avg_ar = sum(x['answer_relevancy']  for x in p2_custom) / len(p2_custom)
        avg_cp = sum(x['context_precision'] for x in p2_custom) / len(p2_custom)
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'f={scores["faithfulness"]:.2f} cr={scores["context_recall"]:.2f} '
              f'ar={scores["answer_relevancy"]:.2f} cp={scores["context_precision"]:.2f} | '
              f'avg: f={avg_f:.3f} cr={avg_cr:.3f} ar={avg_ar:.3f} cp={avg_cp:.3f} | ETA {eta:.1f}m')

print(f'\nSelesai! -> {PHASE2_CUSTOM_PATH}')

Resume: 165/500 selesai dengan 4 metrik.

Tahap 2: Evaluasi 335 sampel baru...

  [  5/335] idx=169 | f=0.67 cr=1.00 ar=1.00 cp=0.50 | avg: f=0.675 cr=0.585 ar=0.733 cp=0.540 | ETA 67.0m
  [ 10/335] idx=174 | f=0.50 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.681 cr=0.597 ar=0.741 cp=0.553 | ETA 62.4m
  [ 15/335] idx=179 | f=0.50 cr=1.00 ar=1.00 cp=0.50 | avg: f=0.687 cr=0.608 ar=0.748 cp=0.557 | ETA 61.6m
  [ 20/335] idx=184 | f=1.00 cr=1.00 ar=1.00 cp=0.50 | avg: f=0.693 cr=0.616 ar=0.753 cp=0.566 | ETA 58.9m
  [ 25/335] idx=189 | f=1.00 cr=1.00 ar=0.67 cp=1.00 | avg: f=0.696 cr=0.626 ar=0.756 cp=0.573 | ETA 58.2m
  [ 30/335] idx=194 | f=1.00 cr=1.00 ar=1.00 cp=0.58 | avg: f=0.700 cr=0.631 ar=0.762 cp=0.577 | ETA 57.8m
  [ 35/335] idx=199 | f=0.50 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.703 cr=0.637 ar=0.768 cp=0.583 | ETA 57.5m
  [ 40/335] idx=204 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.711 cr=0.635 ar=0.774 cp=0.588 | ETA 57.1m
  [ 45/335] idx=209 | f=1.00 cr=1.00 ar=1.00 cp=1.00 | avg: f=0.

## Summary — Hasil Akhir

In [14]:
with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
    p2 = json.load(f)['results']

n      = len(p2)
acc    = sum(r['is_correct']        for r in p2) / n
avg_f  = sum(r['faithfulness']      for r in p2) / n
avg_cr = sum(r['context_recall']    for r in p2) / n
avg_ar = sum(r['answer_relevancy']  for r in p2) / n
avg_cp = sum(r['context_precision'] for r in p2) / n

print('=' * 65)
print(f'  {CONFIG_NAME.upper()} (QR + CR) — {n} sampel')
print(f'  LLM: {LLM_MODEL}')
print('=' * 65)
print(f'  Label Accuracy     : {acc:.1%}')
print(f'  Hallucination Rate : {1-acc:.1%}')
print(f'  Faithfulness       : {avg_f:.4f}')
print(f'  Context Recall     : {avg_cr:.4f}')
print(f'  Answer Relevancy   : {avg_ar:.4f}')
print(f'  Context Precision  : {avg_cp:.4f}')
print(f'  NaN count          : 0')
print('=' * 65)

# Per-label
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = [r for r in p2 if r['ground_truth'] == lbl]
    if sub:
        lbl_acc = sum(r['is_correct']        for r in sub) / len(sub)
        lbl_f   = sum(r['faithfulness']      for r in sub) / len(sub)
        lbl_cr  = sum(r['context_recall']    for r in sub) / len(sub)
        lbl_ar  = sum(r['answer_relevancy']  for r in sub) / len(sub)
        lbl_cp  = sum(r['context_precision'] for r in sub) / len(sub)
        print(f'  {lbl:>5}: acc={lbl_acc:.1%} (n={len(sub)}) | '
              f'faith={lbl_f:.3f} cr={lbl_cr:.3f} ar={lbl_ar:.3f} cp={lbl_cp:.3f}')

print(f'\nBaris tabel skripsi (GPT-4.1-mini):')
print(f'  | Combined QR+CR (GPT-4.1-mini) | {acc:.3f} | {1-acc:.3f} | '
      f'{avg_f:.3f} | {avg_cr:.3f} | {avg_ar:.3f} | {avg_cp:.3f} |')

  COMBINED_OPENAI (QR + CR) — 500 sampel
  LLM: gpt-4.1-mini
  Label Accuracy     : 68.2%
  Hallucination Rate : 31.8%
  Faithfulness       : 0.8102
  Context Recall     : 0.7373
  Answer Relevancy   : 0.8963
  Context Precision  : 0.6758
  NaN count          : 0

Per-label accuracy:
    yes: acc=82.9% (n=275) | faith=0.801 cr=0.745 ar=0.898 cp=0.697
     no: acc=66.7% (n=159) | faith=0.863 cr=0.761 ar=0.917 cp=0.658
  maybe: acc=10.6% (n=66) | faith=0.722 cr=0.647 ar=0.841 cp=0.630

Baris tabel skripsi (GPT-4.1-mini):
  | Combined QR+CR (GPT-4.1-mini) | 0.682 | 0.318 | 0.810 | 0.737 | 0.896 | 0.676 |
